# AI Impact on Job Market - Simplified Task2 Visualization

## Python Implementation (Plotly) - Simplified Version of React/D3 Task2

This notebook provides a **simplified** Python implementation of the React-based Task2 visualization:

### ✅ Core Features Implemented:
- **Bubble Chart**: AI Intensity vs Salary with industry comparison
- **Interactive Year Slider**: Replace D3 drag handles with Plotly widgets.Slider
- **Parallel Coordinates**: Multi-metric time series trends
- **Industry Filtering**: Click legend to show/hide industries
- **Trace Visualization**: Independent chart showing complete evolution paths

### ❌ Features Removed (Simplification):
- Tutorial/Guide system (Spotlight)
- Complex D3 transition animations
- Real-time trace rendering during slider drag
- Drag-to-select year range (replaced with single-year slider)

### Dependencies Required:
```bash
pip install pandas plotly numpy
```

In [48]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 1. Data Loading and Preprocessing

Load and aggregate raw job posting data by year and industry.

In [49]:
# Load CSV data
df_raw = None
try:
    df_raw = pd.read_csv('public/ai_impact_jobs_2010_2025.csv')
    print(f"✅ Data loaded successfully! Shape: {df_raw.shape}")
    print(f"\nColumns: {list(df_raw.columns)}")
except FileNotFoundError:
    print("❌ CSV file not found at 'public/ai_impact_jobs_2010_2025.csv'")
    print("\n⚠️ Generating sample data for demonstration...")
    
    # Generate realistic sample data
    np.random.seed(42)
    industries = ['Tech', 'Finance', 'Healthcare', 'Education', 'Manufacturing', 'Retail']
    years = list(range(2010, 2025))
    
    data = []
    for year in years:
        for industry in industries:
            n_jobs = np.random.randint(50, 200)
            # Simulate realistic trends
            base_salary = {
                'Tech': 85000, 'Finance': 75000, 'Healthcare': 70000,
                'Education': 55000, 'Manufacturing': 60000, 'Retail': 50000
            }
            salary_trend = base_salary[industry] * (1 + 0.03 * (year - 2010))
            ai_trend = 0.3 + 0.04 * (year - 2010) + np.random.uniform(-0.1, 0.1)
            
            data.extend([{
                'posting_year': year,
                'industry': industry,
                'salary_usd': np.random.normal(salary_trend, salary_trend * 0.15),
                'ai_intensity_score': np.clip(ai_trend, 0.1, 0.95),
                'automation_risk_score': np.random.uniform(0.15, 0.75),
            } for _ in range(n_jobs)])
    
    df_raw = pd.DataFrame(data)
    print(f"✅ Sample data generated! Shape: {df_raw.shape}")

display(df_raw.head())

✅ Data loaded successfully! Shape: (5000, 22)

Columns: ['job_id', 'posting_year', 'country', 'region', 'city', 'company_name', 'company_size', 'industry', 'job_title', 'seniority_level', 'ai_mentioned', 'ai_keywords', 'ai_intensity_score', 'core_skills', 'ai_skills', 'salary_usd', 'salary_change_vs_prev_year_percent', 'automation_risk_score', 'reskilling_required', 'ai_job_displacement_risk', 'job_description_embedding_cluster', 'industry_ai_adoption_stage']


,job_id,posting_year,country,region,city,company_name,company_size,industry,job_title,seniority_level,...,ai_intensity_score,core_skills,ai_skills,salary_usd,salary_change_vs_prev_year_percent,automation_risk_score,reskilling_required,ai_job_displacement_risk,job_description_embedding_cluster,industry_ai_adoption_stage
0,836b4774-702e-49ef-93d3-2f255ce1e910,2018,Brazil,South America,London,NextGen Technologies,Small,Education,Policy Analyst,Lead,...,0.81,"Research, Project Management, Business Analysis",reinforcement learning,61586,12.68,0.11,True,Low,14,Growing
1,43699e93-7b15-4728-a4c6-9e41ff438a25,2015,UAE,Middle East,Singapore,Future Solutions,Medium,Energy,Data Scientist,Executive,...,0.04,"Research, SQL, Business Analysis, Python, Clou...",NaN,62045,-3.98,0.71,False,High,19,Emerging
2,fc9d1854-3cbf-4bab-90df-77304dfc59df,2016,Nepal,South Asia,Sydney,Future Analytics,Startup,Finance,Product Manager,Junior,...,0.15,"Statistics, Project Management, Cloud Computin...",NaN,27035,3.55,0.86,False,High,2,Emerging
3,05c1c7d3-2add-4919-91eb-f6c78bfe23d1,2015,Spain,Europe,Nairobi,Global Technologies,Large,Government,Data Scientist,Mid,...,0.19,"Cloud Computing, SQL, Project Management, Comm...",NaN,72894,-2.80,0.70,False,Low,15,Emerging
4,5e739937-d1b0-44d7-935c-7ebb3fc1f6e8,2014,Taiwan,East Asia,Sydney,Future Technologies,Small,Manufacturing,ML Engineer,Lead,...,0.11,"SQL, Python, Communication, Software Engineeri...",NaN,57215,0.85,0.87,False,High,13,Emerging


In [50]:
# Aggregate data by year and industry (matching React's d3.rollups logic)
df_agg = df_raw.groupby(['posting_year', 'industry']).agg(
    avgSalary=('salary_usd', 'mean'),
    avgAIIntensity=('ai_intensity_score', 'mean'),
    avgAutomationRisk=('automation_risk_score', 'mean'),
    jobCount=('industry', 'size')
).reset_index()

# Rename columns to match React implementation
df_agg = df_agg.rename(columns={'posting_year': 'year'})

print(f"✅ Aggregated data shape: {df_agg.shape}")
print(f"📅 Year range: {df_agg['year'].min()} - {df_agg['year'].max()}")
print(f"🏭 Industries: {list(df_agg['industry'].unique())}")
print(f"\nSample aggregated data:")
display(df_agg.head(10))

✅ Aggregated data shape: (144, 6)
📅 Year range: 2010 - 2025
🏭 Industries: ['Agriculture', 'Education', 'Energy', 'Finance', 'Government', 'Healthcare', 'Manufacturing', 'Retail', 'Tech']

Sample aggregated data:


,year,industry,avgSalary,avgAIIntensity,avgAutomationRisk,jobCount
0,2010,Agriculture,47219.756098,0.105122,0.732927,41
1,2010,Education,48564.666667,0.107667,0.750000,30
2,2010,Energy,51022.413793,0.154483,0.728966,29
3,2010,Finance,55057.081081,0.205405,0.638919,37
4,2010,Government,47817.085106,0.172979,0.690000,47
5,2010,Healthcare,43619.054054,0.136486,0.721351,37
6,2010,Manufacturing,42342.707317,0.136098,0.698049,41
7,2010,Retail,47180.192308,0.149231,0.717692,26
8,2010,Tech,46424.820513,0.242564,0.606410,39
9,2011,Agriculture,46531.906250,0.097187,0.748125,32


## 2. Interactive Bubble Chart with Year Slider

This chart replaces the React D3 drag-handle interaction with a **Plotly Slider widget**.

### Key Features:
- **X-axis**: AI Intensity Score
- **Y-axis**: Average Salary (USD)
- **Bubble size**: Job Count (scaled with sqrt)
- **Bubble opacity**: Automation Risk (40%-80%)
- **Solid circles**: Selected year data
- **Arrow lines**: Show evolution from previous year
- **Slider**: Interactively change the displayed year

### How it works:
Instead of D3's complex drag event handlers, we use Plotly's `frames` and `sliders` to pre-compute all year states. When the user drags the slider, Plotly instantly switches between pre-rendered frames.

In [51]:
# Industry colors matching React implementation exactly
INDUSTRY_COLORS = {
    'Tech': '#dc2626',
    'Finance': '#d97706',
    'Healthcare': '#db2777',
    'Education': '#0891b2',
    'Manufacturing': '#7c3aed',
    'Retail': '#ea580c',
    'Agriculture': '#16a34a',
    'Energy': '#0d9488',
    'Government': '#f59e0b',
}

print("✅ Industry color palette defined")
for industry, color in INDUSTRY_COLORS.items():
    print(f"   {industry}: {color}")

✅ Industry color palette defined
   Tech: #dc2626
   Finance: #d97706
   Healthcare: #db2777
   Education: #0891b2
   Manufacturing: #7c3aed
   Retail: #ea580c
   Agriculture: #16a34a
   Energy: #0d9488
   Government: #f59e0b


In [52]:
def create_interactive_bubble_chart(df):
    """
    Create interactive bubble chart with Plotly slider.
    
    This replaces the React D3 drag-handle logic with a simpler slider approach.
    Instead of selecting two years simultaneously, users slide through individual years
    and see arrows showing evolution from the previous year.
    """
    
    years = sorted(df['year'].unique())
    industries = sorted(df['industry'].unique())
    
    # Create figure
    fig = go.Figure()
    
    # Pre-compute frames for each year
    frames = []
    for year_idx, year in enumerate(years):
        frame_data = df[df['year'] == year].copy()
        prev_year = year - 1 if year > years[0] else None
        
        frame_traces = []
        
        # Add arrow lines from previous year (if exists)
        if prev_year:
            prev_data = df[df['year'] == prev_year]
            common_industries = set(frame_data['industry']).intersection(set(prev_data['industry']))
            
            for industry in common_industries:
                curr_row = frame_data[frame_data['industry'] == industry].iloc[0]
                prev_row = prev_data[prev_data['industry'] == industry].iloc[0]
                color = INDUSTRY_COLORS.get(industry, '#64748b')
                
                frame_traces.append(go.Scatter(
                    x=[prev_row['avgAIIntensity'], curr_row['avgAIIntensity']],
                    y=[prev_row['avgSalary'], curr_row['avgSalary']],
                    mode='lines',
                    line=dict(color=color, width=2.5, dash='dot'),
                    opacity=0.6,
                    showlegend=False,
                    hoverinfo='skip',
                    visible=True  # Always visible in frames
                ))
        
        # Add bubbles for current year
        # CRITICAL FIX: Must add traces for ALL industries in ALL frames for legend to work
        for industry in industries:
            color = INDUSTRY_COLORS.get(industry, '#64748b')
            
            if industry in frame_data['industry'].values:
                row = frame_data[frame_data['industry'] == industry].iloc[0]
                size = float(np.sqrt(row['jobCount'])) * 2.5  # Scale factor
                opacity = float(0.4 + row['avgAutomationRisk'] * 0.6)
                
                frame_traces.append(go.Scatter(
                    x=[float(row['avgAIIntensity'])],
                    y=[float(row['avgSalary'])],
                    mode='markers',
                    marker=dict(
                        size=size,
                        color=color,
                        opacity=opacity,
                        line=dict(color='white', width=2)
                    ),
                    name=industry,
                    text=f"{industry}<br>Year: {year}<br>Salary: ${row['avgSalary']:.0f}<br>AI Intensity: {row['avgAIIntensity']:.2f}<br>Jobs: {int(row['jobCount'])}",
                    hoverinfo='text',
                    legendgroup=industry,
                    showlegend=(year_idx == 0)  # Show legend only in first frame
                ))
            else:
                # Add invisible placeholder for this industry to maintain trace order
                frame_traces.append(go.Scatter(
                    x=[None],
                    y=[None],
                    mode='markers',
                    marker=dict(size=0, color=color),
                    name=industry,
                    legendgroup=industry,
                    showlegend=False,
                    hoverinfo='skip',
                    opacity=0
                ))
        
        frames.append(go.Frame(data=frame_traces, name=str(year)))
    
    # Add initial frame (first year) to main figure
    fig.frames = frames
    fig.add_traces(frames[0].data)
    
    # Create slider steps
    sliders = [{
        'active': 0,
        'currentvalue': {'prefix': 'Year: ', 'font': {'size': 16}},
        'pad': {'t': 80},  # Increased padding to avoid overlap
        'len': 0.9,
        'x': 0.05,
        'xanchor': 'left',
        'y': 0.02,  # Position at bottom
        'yanchor': 'bottom',
        'steps': [
            {
                'args': [[str(year)], {'frame': {'duration': 300, 'redraw': True}, 'mode': 'immediate'}],
                'label': str(year),
                'method': 'animate'
            } for year in years
        ]
    }]
    
    # Update layout
    fig.update_layout(
        title=dict(
            text='AI Impact on Industries: Interactive Time Series',
            font=dict(size=20),
            x=0.5
        ),
        xaxis=dict(
            title='AI Intensity Score',
            gridcolor='#e2e8f0',
            zeroline=False,
            range=[df['avgAIIntensity'].min() - 0.05, df['avgAIIntensity'].max() + 0.05]
        ),
        yaxis=dict(
            title='Average Salary (USD)',
            gridcolor='#e2e8f0',
            tickformat='$,.0f',
            zeroline=False
        ),
        template='plotly_white',
        width=950,
        height=650,
        margin=dict(b=150),  # Extra bottom margin for controls
        legend=dict(
            title='Industries (Click to toggle)',
            orientation='h',
            yanchor='bottom',
            y=-0.25,
            xanchor='center',
            x=0.5
        ),
        hoverlabel=dict(bgcolor='white', font_size=12),
        updatemenus=[{
            'type': 'buttons',
            'showactive': False,
            'x': 0.05,
            'y': -0.12,  # Position above slider
            'xanchor': 'left',
            'yanchor': 'bottom',
            'pad': {'r': 10, 't': 10},
            'buttons': [
                {
                    'label': '▶ Play',
                    'method': 'animate',
                    'args': [None, {'frame': {'duration': 500, 'redraw': True}, 'fromcurrent': True}]
                },
                {
                    'label': '⏸ Pause',
                    'method': 'animate',
                    'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}]
                }
            ]
        }],
        sliders=sliders
    )
    
    return fig

# Create and display the interactive bubble chart
print("🎨 Generating interactive bubble chart with slider...")
fig_bubble_slider = create_interactive_bubble_chart(df_agg)
fig_bubble_slider.show()

🎨 Generating interactive bubble chart with slider...


## 3. Parallel Coordinates - Multi-Metric Trends

Shows time series evolution for all 4 key metrics across industries.

### Metrics Displayed:
1. **AI Intensity Score** - How much AI is integrated into the industry
2. **Average Salary (USD)** - Compensation trends over time
3. **Automation Risk Score** - Probability of job automation
4. **Job Count** - Market size and growth

This replaces the React implementation's scrollable parallel coordinates with a compact subplot layout.

In [53]:
def create_parallel_coordinates_chart(df):
    """
    Create 4-subplot parallel coordinates showing metric trends over time.
    
    Each subplot shows one metric's evolution across all industries.
    Click legend items to toggle industry visibility.
    """
    
    # Create subplots
    fig = make_subplots(
        rows=4, cols=1,
        subplot_titles=(
            '📊 AI Intensity Score',
            '💰 Average Salary (USD)',
            '⚠️ Automation Risk Score',
            '👥 Job Count'
        ),
        vertical_spacing=0.08,
        shared_xaxes=True
    )
    
    industries = sorted(df['industry'].unique())
    years = sorted(df['year'].unique())
    
    metrics = [
        ('avgAIIntensity', ':.2f', ''),
        ('avgSalary', '$,.0f', ''),
        ('avgAutomationRisk', '.0%', ''),
        ('jobCount', ',', '')
    ]
    
    for row_idx, (metric, fmt, suffix) in enumerate(metrics, 1):
        for industry in industries:
            industry_data = df[df['industry'] == industry].sort_values('year')
            color = INDUSTRY_COLORS.get(industry, '#64748b')
            
            fig.add_trace(
                go.Scatter(
                    x=industry_data['year'],
                    y=industry_data[metric],
                    mode='lines+markers',
                    name=industry,
                    line=dict(color=color, width=2.5),
                    marker=dict(size=6, color=color, line=dict(width=1, color='white')),
                    legendgroup=industry,
                    showlegend=(row_idx == 1)  # Show legend only in first subplot
                ),
                row=row_idx,
                col=1
            )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text='Multi-Metric Time Series by Industry (2010-2024)',
            font=dict(size=18),
            x=0.5
        ),
        template='plotly_white',
        width=950,
        height=900,
        showlegend=True,
        legend=dict(
            title='Industries (Click to toggle)',
            orientation='h',
            yanchor='bottom',
            y=-0.05,
            xanchor='center',
            x=0.5
        ),
        hoverlabel=dict(bgcolor='white', font_size=11)
    )
    
    # Update axes for all subplots
    for i in range(1, 5):
        fig.update_yaxes(
            gridcolor='#e2e8f0',
            zeroline=False,
            row=i, col=1
        )
    
    fig.update_xaxes(
        gridcolor='#e2e8f0',
        tickformat='d',
        tickvals=years,
        row=4, col=1
    )
    
    return fig

# Create and display parallel coordinates
print("📈 Generating parallel coordinates chart...")
fig_parallel = create_parallel_coordinates_chart(df_agg)
fig_parallel.show()

📈 Generating parallel coordinates chart...


## 4. Industry Evolution Trace Visualization

This chart shows **complete evolution paths** for all industries across the entire time span.

### Key Differences from React Version:
- ❌ **No real-time rendering**: Unlike React's dynamic trace that updates during drag, this is a static view of the full timeline
- ✅ **Complete paths**: Shows every year's data point connected by lines
- ✅ **Ghost markers**: Dashed circles at each historical position indicate job count changes

### Use Case:
Use this to understand long-term industry trajectories without needing to manually scrub through years.

In [54]:
def create_evolution_trace_chart(df, start_year=None, end_year=None):
    """
    Create comprehensive evolution trace showing complete industry paths.
    
    This is an INDEPENDENT visualization - not tied to slider interaction.
    Shows how industries move through AI-Salary space over the selected period.
    """
    
    if start_year is None:
        start_year = df['year'].min()
    if end_year is None:
        end_year = df['year'].max()
    
    fig = go.Figure()
    
    # Filter data for trace period
    trace_data = df[(df['year'] >= start_year) & (df['year'] <= end_year)]
    industries = sorted(trace_data['industry'].unique())
    
    for industry in industries:
        industry_path = trace_data[trace_data['industry'] == industry].sort_values('year')
        color = INDUSTRY_COLORS.get(industry, '#64748b')
        
        # Add evolution path (solid line connecting positions)
        fig.add_trace(go.Scatter(
            x=industry_path['avgAIIntensity'],
            y=industry_path['avgSalary'],
            mode='lines+markers',
            name=industry,
            line=dict(color=color, width=2.5),
            marker=dict(
                size=10,
                color=color,
                line=dict(width=2, color='white')
            ),
            text=[f"{industry}<br>Year: {year}<br>AI: {ai:.2f}<br>Salary: ${sal:,.0f}" 
                  for year, ai, sal in zip(industry_path['year'], 
                                          industry_path['avgAIIntensity'],
                                          industry_path['avgSalary'])],
            hoverinfo='text'
        ))
        
        # Add ghost markers (dashed circles showing job count at each year)
        for _, row in industry_path.iterrows():
            size = np.sqrt(row['jobCount']) * 2
            fig.add_trace(go.Scatter(
                x=[row['avgAIIntensity']],
                y=[row['avgSalary']],
                mode='markers',
                marker=dict(
                    size=size * 1.5,
                    color='rgba(0,0,0,0)',
                    line=dict(color=color, width=1.5, dash='dash')
                ),
                showlegend=False,
                hoverinfo='skip',
                opacity=0.5
            ))
    
    fig.update_layout(
        title=dict(
            text=f'Industry Evolution Paths ({start_year}-{end_year})',
            font=dict(size=18),
            x=0.5
        ),
        xaxis=dict(
            title='AI Intensity Score →',
            gridcolor='#e2e8f0',
            zeroline=False
        ),
        yaxis=dict(
            title='Average Salary (USD) →',
            gridcolor='#e2e8f0',
            tickformat='$,.0f',
            zeroline=False
        ),
        template='plotly_white',
        width=950,
        height=650,
        showlegend=True,
        legend=dict(
            title='Industries',
            orientation='h',
            yanchor='bottom',
            y=-0.15,
            xanchor='center',
            x=0.5
        ),
        hoverlabel=dict(bgcolor='white', font_size=12)
    )
    
    # Add annotation explaining the visualization
    fig.add_annotation(
        text="💡 Solid lines = Position trajectory | Dashed circles = Job count size",
        xref="paper", yref="paper",
        x=0.5, y=-0.22,
        showarrow=False,
        font=dict(size=11, color="#64748b")
    )
    
    return fig

# Create and display trace visualization
print("🔍 Generating evolution trace chart...")
fig_trace = create_evolution_trace_chart(df_agg)
fig_trace.show()

🔍 Generating evolution trace chart...


ValueError: Invalid property specified for object of type plotly.graph_objs.scatter.marker.Line: 'dash'

Did you mean "cauto"?

    Valid properties:
        autocolorscale
            Determines whether the colorscale is a default palette
            (`autocolorscale: true`) or the palette determined by
            `marker.line.colorscale`. Has an effect only if in
            `marker.line.color` is set to a numerical array. In
            case `colorscale` is unspecified or `autocolorscale` is
            true, the default palette will be chosen according to
            whether numbers in the `color` array are all positive,
            all negative or mixed.
        cauto
            Determines whether or not the color domain is computed
            with respect to the input data (here in
            `marker.line.color`) or the bounds set in
            `marker.line.cmin` and `marker.line.cmax` Has an effect
            only if in `marker.line.color` is set to a numerical
           

## 5. Dynamic Leaderboard Table

Replicates the React sidebar ranking functionality with pandas styling.

### Features:
- Sort by any metric (Salary, AI Intensity, Risk, Job Count)
- Toggle ascending/descending order
- Color-coded industry indicators
- Formatted values matching React display

In [55]:
def create_leaderboard_table(df, metric='avgSalary', year=None, ascending=False):
    """
    Create styled leaderboard table showing industry rankings.
    
    Args:
        df: Aggregated dataframe
        metric: Metric to rank by ('avgSalary', 'avgAIIntensity', 'avgAutomationRisk', 'jobCount')
        year: Specific year to display (default: latest year)
        ascending: Sort order (False = descending/highest first)
    """
    
    if year is None:
        year = df['year'].max()
    
    # Filter by year
    year_data = df[df['year'] == year].copy()
    
    # Add color column
    year_data['color'] = year_data['industry'].map(lambda x: INDUSTRY_COLORS.get(x, '#64748b'))
    
    # Sort by metric
    year_data = year_data.sort_values(metric, ascending=ascending)
    year_data['rank'] = range(1, len(year_data) + 1)
    
    # Format values based on metric type
    if metric == 'avgSalary':
        year_data['formatted'] = year_data[metric].apply(lambda x: f'${x/1000:.1f}k')
        metric_label = 'Avg Salary'
    elif metric == 'avgAutomationRisk':
        year_data['formatted'] = year_data[metric].apply(lambda x: f'{x*100:.1f}%')
        metric_label = 'Auto Risk'
    elif metric == 'avgAIIntensity':
        year_data['formatted'] = year_data[metric].apply(lambda x: f'{x:.2f}')
        metric_label = 'AI Intensity'
    else:  # jobCount
        year_data['formatted'] = year_data[metric].apply(lambda x: f'{x:,}')
        metric_label = 'Job Count'
    
    # Create display dataframe
    display_df = year_data[['rank', 'industry', 'formatted', 'color']].copy()
    display_df.columns = ['Rank', 'Industry', metric_label, 'Color']
    
    # Apply styling
    styled = display_df.style.apply(
        lambda row: [
            f'background-color: {row["Color"]}22' if col == 'Industry' else '',
            f'color: {row["Color"]}; font-weight: bold' if col == 'Industry' else ''
        ] + [''] * (len(row) - 2),
        axis=1
    )
    
    return styled

# Display multiple leaderboard views
print("\n" + "="*60)
print("📊 LEADERBOARD TABLES")
print("="*60)

latest_year = df_agg['year'].max()
print(f"\n🏆 Top Industries by Average Salary ({latest_year}):")
display(create_leaderboard_table(df_agg, metric='avgSalary', year=latest_year, ascending=False))

print(f"\n🤖 Highest AI Intensity ({latest_year}):")
display(create_leaderboard_table(df_agg, metric='avgAIIntensity', year=latest_year, ascending=False))

print(f"\n⚠️ Highest Automation Risk ({latest_year}):")
display(create_leaderboard_table(df_agg, metric='avgAutomationRisk', year=latest_year, ascending=False))


📊 LEADERBOARD TABLES

🏆 Top Industries by Average Salary (2025):


NameError: name 'col' is not defined


🤖 Highest AI Intensity (2025):


NameError: name 'col' is not defined


⚠️ Highest Automation Risk (2025):


NameError: name 'col' is not defined

## 6. Summary & Implementation Notes

### ✅ What Python/Plotly Achieves:

1. **Data Processing Excellence**
   - `pandas` provides superior aggregation capabilities compared to D3
   - Easy groupby operations replace complex `d3.rollups`

2. **Native Interactivity**
   - Hover tooltips work out-of-the-box
   - Zoom, pan, and box select built-in
   - Legend click-to-toggle requires zero custom code

3. **Slider Widget**
   - Replaces complex D3 drag handlers with simple `frames` + `sliders`
   - Smooth transitions between years
   - Play/Pause buttons for automatic animation

4. **Export Flexibility**
   - Save as standalone HTML files
   - Export to PNG/PDF for reports
   - Embed in web apps or Jupyter notebooks

### ⚠️ Limitations vs React/D3:

1. **Custom Animations**
   - Plotly's transitions are less customizable than D3's `.transition()`
   - Cannot easily implement complex easing functions

2. **State Management**
   - React's component state allows complex interactions (e.g., multi-select, compare modes)
   - Plotly widgets are more limited in stateful interactions

3. **UI Customization**
   - CSS styling in React offers pixel-perfect control
   - Plotly's layout options are more constrained

4. **Complex Overlays**
   - Tutorial spotlight system difficult to replicate in Jupyter
   - Custom tooltip positioning less flexible

### 🔄 State Update Logic Comparison:

**React/D3 Approach:**
```javascript
// Complex event-driven state management
const dragLeft = d3.drag()
  .on('drag', (event) => {
    setCompareYears(prev => {
      const newYear = calculateYearFromPosition(event.x);
      return { ...prev, left: newYear };
    });
  });
```

**Python/Plotly Approach:**
```python
# Pre-computed frames with slider widget
frames = [create_frame_for_year(year) for year in years]
slider = dict(steps=[{'args': [[str(year)]], 'method': 'animate'} for year in years])
# No manual event handling needed!
```

**Key Insight:** Plotly's declarative approach pre-computes all states, while React's imperative approach calculates states on-demand during user interaction.

### 💡 Recommendation:

| Use Case | Recommended Tool |
|----------|------------------|
| Data exploration & analysis | **Python/Plotly** |
| Static reports & presentations | **Python/Plotly** |
| Quick prototyping | **Python/Plotly** |
| Production dashboards | **React/D3** |
| Complex animations | **React/D3** |
| Custom UI/UX requirements | **React/D3** |

---
## Bonus: Export Charts to Standalone HTML

Generate self-contained HTML files that can be shared or embedded in websites.

These files include all necessary JavaScript and data - no server required!

In [ ]:
# Export all charts to HTML
output_files = {
    'task2_interactive_bubble.html': fig_bubble_slider,
    'task2_parallel_coords.html': fig_parallel,
    'task2_evolution_trace.html': fig_trace
}

for filename, fig in output_files.items():
    fig.write_html(filename, include_plotlyjs='cdn', full_html=True)
    print(f"✅ Exported: {filename}")

print("\n📁 Generated Files:")
for filename in output_files.keys():
    print(f"   • {filename}")

print("\n💡 Tip: Open these HTML files in any modern browser to interact with the charts!")